In [0]:
CREATE OR REPLACE VIEW databricks_ws.gold.mart_genre_performance AS
SELECT
    g.genre_id,
    g.genre_name,
    m.movie_id,
    m.title,
    t.release_year AS year_release_date,
    f.budget,
    f.revenue,

    CASE
        WHEN f.budget IS NULL OR f.budget <= 0 THEN 'Sin datos'
        WHEN f.budget < 10000000 THEN 'Bajo'
        WHEN f.budget < 50000000 THEN 'Medio'
        WHEN f.budget < 100000000 THEN 'Alto'
        ELSE 'Muy alto'
    END AS budget_range,

    CASE
        WHEN f.budget > 0 AND f.revenue > 0
        THEN ROUND(f.revenue / f.budget, 2)
    END AS revenue_multiple,

    CASE
        WHEN f.budget > 0 AND f.revenue > 0
        THEN CASE WHEN f.revenue >= f.budget * 2 THEN 1 ELSE 0 END
    END AS reached_2x

FROM databricks_ws.gold.fact_movies f

JOIN databricks_ws.gold.dim_movies m
    ON f.movie_id = m.movie_id

JOIN databricks_ws.gold.dim_time t
    ON f.time_id = t.time_id

JOIN databricks_ws.gold.bridge_movie_genres mg
    ON f.movie_id = mg.movie_id

JOIN databricks_ws.gold.dim_genres g
    ON mg.genre_id = g.genre_id;

In [0]:

SELECT
    genre_name,
    COUNT(DISTINCT movie_id) AS total_movies,
    ROUND(MEDIAN(revenue_multiple), 2) AS median_multiple,
    ROUND(AVG(reached_2x) * 100, 2) AS success_rate_2x

FROM databricks_ws.gold.mart_genre_performance

GROUP BY genre_id, genre_name
ORDER BY success_rate_2x DESC;

genre_name,total_movies,median_multiple,success_rate_2x
Adventure,773,1.91,48.12
Animation,224,1.89,47.77
Fantasy,413,1.77,44.79
Science Fiction,526,1.55,44.11
Family,507,1.62,43.59
War,137,1.63,43.07
Action,1115,1.55,42.69
Horror,514,1.3,41.83
Mystery,343,1.3,40.23
Thriller,1260,1.3,39.13
